In [ ]:
# ===== 0. PENGATURAN =====
PROJECT_ID = "ISI-PROJECT-ID-KAMU"
DATASET    = "porto_proc"
LOCATION   = "asia-southeast2"
CSV_DIR    = "csv"

In [ ]:
# ===== 1. LOGIN, DATASET, DAN FILE CSV =====
import os, glob, zipfile
from google.colab import auth, files
auth.authenticate_user()

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

ds = bigquery.Dataset(f"{PROJECT_ID}.{DATASET}")
ds.location = LOCATION
ds.default_table_expiration_ms = 60 * 24 * 3600 * 1000
client.create_dataset(ds, exists_ok=True)

if not os.path.exists(os.path.join(CSV_DIR, "po_line.csv")):
    up = files.upload()
    for n in up:
        if n.endswith(".zip"): zipfile.ZipFile(n).extractall("data_csv")
    CSV_DIR = os.path.dirname(glob.glob("data_csv/**/po_line.csv", recursive=True)[0])
print("folder CSV:", CSV_DIR)

Saving data_csv.zip to data_csv (1).zip
folder CSV: data_csv/csv


In [ ]:
# ===== 2. SKEMA TABEL (urutan kolom = urutan kolom di CSV) =====
SKEMA = {
    "supplier": [("supplier_id", "STRING"), ("supplier_name", "STRING"), ("kategori_utama", "STRING"),
                 ("tipe_supplier", "STRING"), ("terms_hari", "INT64"), ("tarif_ppn", "FLOAT64"), ("status_pkp", "STRING")],
    "item": [("item_id", "STRING"), ("item_name", "STRING"), ("kategori", "STRING"), ("uom", "STRING"), ("harga_acuan", "FLOAT64")],
    "param_tolerance": [("kategori", "STRING"), ("tol_gr_vs_po", "FLOAT64"), ("tol_inv_vs_gr", "FLOAT64"), ("tol_harga", "FLOAT64")],
    "po_line": [("po_line_id", "STRING"), ("po_id", "STRING"), ("line_no", "INT64"), ("po_date", "DATE"), ("supplier_id", "STRING"),
                ("item_id", "STRING"), ("qty_po", "FLOAT64"), ("uom", "STRING"), ("ekor_po", "INT64"), ("harga_po", "FLOAT64"),
                ("nilai_po", "FLOAT64"), ("terms_hari", "INT64"), ("tgl_kirim_diharapkan", "DATE")],
    "gr_line": [("gr_line_id", "STRING"), ("gr_no", "STRING"), ("gr_date", "DATE"), ("po_line_id", "STRING"), ("supplier_id", "STRING"),
                ("qty_diterima", "FLOAT64"), ("ekor_diterima", "INT64"), ("surat_jalan", "STRING"), ("lokasi_terima", "STRING"),
                ("catatan", "STRING")],
    "inv_line": [("inv_line_id", "STRING"), ("inv_no", "STRING"), ("inv_date", "DATE"), ("tgl_input", "DATE"), ("supplier_id", "STRING"),
                 ("po_line_id", "STRING"), ("qty_faktur", "FLOAT64"), ("harga_faktur", "FLOAT64"), ("jumlah_faktur", "FLOAT64"),
                 ("ppn_faktur", "FLOAT64"), ("total_faktur", "FLOAT64")],
}
BARIS_HARUS = {"po_line": 112, "gr_line": 116, "inv_line": 110, "item": 16, "supplier": 9, "param_tolerance": 4}

In [ ]:
# ===== 3. LOAD JOB (WRITE_TRUNCATE: aman dijalankan berulang) =====
for t, cols in SKEMA.items():
    cfg = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,
        schema=[bigquery.SchemaField(n, ty) for n, ty in cols],
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)
    with open(f"{CSV_DIR}/{t}.csv", "rb") as f:
        client.load_table_from_file(f, f"{PROJECT_ID}.{DATASET}.{t}", job_config=cfg).result()
    n = client.get_table(f"{PROJECT_ID}.{DATASET}.{t}").num_rows
    status = "OK" if n == BARIS_HARUS[t] else f"PERIKSA (seharusnya {BARIS_HARUS[t]})"
    print(f"{t:16s} {n:4d} baris  {status}")

supplier            9 baris  OK
item               16 baris  OK
param_tolerance     4 baris  OK
po_line           112 baris  OK
gr_line             0 baris  PERIKSA (seharusnya 116)
inv_line          110 baris  OK
